# 2D Heliosphere Model


# Setup


## Imports


In [36]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/helio_n_matplotlib")

cwd = Path.cwd().resolve()
project_root = None
for candidate in (cwd, *cwd.parents, Path("/home/smdc/helio-n")):
    if (candidate / "Library").exists() and (candidate / "Config").exists():
        project_root = candidate
        break
assert (
    project_root is not None
), "Could not locate the helio-n project root for notebook imports."
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
from ipywidgets import interact, widgets

from Library.SW.Ballistic import (
    init_accumulators,
    postprocess_max_field,
    prepare_seed_inputs,
    run_bulk_propagation,
)
from Library.SW.Config import (
    load_ballistic_spec,
    load_sw_runtime_spec,
)
from Library.SW.Coords import (
    build_grid_axes,
    build_packet_geometry,
    build_transport_state,
    compute_rotation_state,
)
from Library.SW.Inputs import (
    DEFAULT_SQL_CONNECTION,
    DEFAULT_SQL_QUERY,
    build_ace_earth_swx_frame,
    build_model_input_series,
    load_ace_earth_frame,
    load_sw_input_frame,
    v_from_area,
)
from Library.SW.Visualization import (
    build_satellite_comparison_frame,
    export_polar_animation,
    plot_polar_snapshot,
)
from Models.CH_SW_Correspondence.Shugay_Slow_SW import load as load_shugay_slow_sw


## Global Parameters


In [37]:
empirical = load_shugay_slow_sw()
ballistic = load_ballistic_spec()
runtime = load_sw_runtime_spec()

# Keep short aliases for the repeated physics terms used throughout the notebook.
# superresolution_enabled = bool(ballistic["superresolution_enabled"])
superresolution_enabled = False
time_step_minutes = (
    int(ballistic["superresolution_step_minutes"])
    if superresolution_enabled
    else int(ballistic["base_time_step_minutes"])
)
time_step_hours = float(time_step_minutes) / 60.0
time_freq = f"{int(time_step_minutes)}min"
phi_step_minutes = ballistic["phi_step_minutes"]

cr_days = ballistic["cr_days"]
r0 = ballistic["r0"]
rSolar = ballistic["r_solar_km"]
R_max = ballistic["r_max"]
horizon_hours = ballistic["horizon_hours"]
use_swept_cell = ballistic["use_swept_cell"]
field_half_width_h = ballistic["field_half_width_h"]
use_cr_reset = ballistic["use_cr_reset"]
memory_guard_enabled = ballistic["memory_guard_enabled"]
simulation_pad_days = ballistic["simulation_pad_days"]

dense_memory_budget_gb = runtime["dense_memory_budget_gb"]
max_seed_batch = runtime["max_seed_batch"]
post_chunk_t = runtime["post_chunk_t"]

print(
    "Shugay_Slow_SW.py:",
    empirical.source_path,
    "| Ballistic.json:",
    ballistic["json_path"],
)
print(
    f"Machine.json[{runtime['machine_name']!r}].sw:",
    runtime["machine_json_path"],
)
print(
    "superresolution_enabled:",
    superresolution_enabled,
    "| dt(min):",
    time_step_minutes,
    "| freq:",
    time_freq,
)
print("phi_step_minutes:", phi_step_minutes)
print(
    "dense_memory_budget_gb:",
    dense_memory_budget_gb,
    "| max_seed_batch:",
    max_seed_batch,
)


Shugay_Slow_SW.py: /Users/aosh/Developer/helio-n/Models/CH_SW_Correspondence/Shugay_Slow_SW.py | Ballistic.json: /Users/aosh/Developer/helio-n/Config/SW/Ballistic.json
Machine.json['risc1'].sw: /Users/aosh/Developer/helio-n/Config/Machine.json
superresolution_enabled: False | dt(min): 60 | freq: 60min
phi_step_minutes: 120
dense_memory_budget_gb: 10.0 | max_seed_batch: 256


## Data Loading And Input Speed Series


In [38]:
start_dt = pd.Timestamp("2016-12-15")
end_dt = pd.Timestamp("2017-02-15")

input_source = "parquet"
input_parquet_path = Path("Data/CH Area.parquet")
sql_query = DEFAULT_SQL_QUERY
sql_connection_kwargs = DEFAULT_SQL_CONNECTION.copy()

In [39]:
df_sdo_sw_test = load_sw_input_frame(
    start_dt=start_dt,
    end_dt=end_dt,
    source=input_source,
    input_parquet_path=input_parquet_path,
    query=sql_query,
    connection_kwargs=sql_connection_kwargs,
)
print(
    "Loaded propagation input from:",
    input_parquet_path if input_source == "parquet" else "SQL query",
)
print("Rows in filtered input:", len(df_sdo_sw_test))
df_sdo_sw_test.head()

Loaded propagation input from: Data/CH Area.parquet
Rows in filtered input: 1473


,dt,sw_speed_1,ch_area_1,sw_speed_2,ch_area_2,ch_relative_area,forecast_sw_speed
0,2016-12-15 15:00:00,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-12-15 16:00:00,NaN,0.33,NaN,NaN,0.33,NaN
2,2016-12-15 17:00:00,NaN,0.31,NaN,NaN,0.31,NaN
3,2016-12-15 18:00:00,315.60137,0.30,NaN,NaN,0.30,315.60137
4,2016-12-15 19:00:00,315.09968,0.30,NaN,NaN,0.30,315.09968


## Empirical SW Speed Model


In [40]:
sample_time = pd.date_range("2017-01-01 00:00:00", periods=3, freq="1h")
sample_area = pd.Series([0.0, 0.33, 0.66], index=sample_time, name="ch_relative_area")
pd.DataFrame(
    {
        "ch_relative_area": sample_area,
        "v_empirical": v_from_area(sample_area, empirical, t=sample_time),
    }
)


,ch_relative_area,v_empirical
2017-01-01 00:00:00,0.00,302.522080
2017-01-01 01:00:00,0.33,393.818825
2017-01-01 02:00:00,0.66,440.294680


## Simulation Input Preparation


In [41]:
prepared_inputs = build_model_input_series(
    sdo_input_df=df_sdo_sw_test,
    empirical=empirical,
    superresolution_enabled=superresolution_enabled,
    time_freq=time_freq,
    simulation_pad_days=simulation_pad_days,
)

sdo_input_df = prepared_inputs["sdo_input_df"]
df_v = prepared_inputs["df_v"]
df_ch_area_hourly = prepared_inputs["df_ch_area_hourly"]
df_ch_area = prepared_inputs["df_ch_area"]
sim_start = prepared_inputs["sim_start"]
sim_end = prepared_inputs["sim_end"]

print("Rows in df_v:", len(df_v), "| cadence:", time_freq)
print(
    "Rows in df_ch_area_hourly:",
    len(df_ch_area_hourly),
    "| stepwise series rows:",
    len(df_ch_area),
)
df_v.head()

Rows in df_v: 1472 | cadence: 60min
Rows in df_ch_area_hourly: 1472 | stepwise series rows: 1472


,v
time,
2016-12-15 16:00:00,421.807197
2016-12-15 17:00:00,418.478467
2016-12-15 18:00:00,412.470557
2016-12-15 19:00:00,411.704446
2016-12-15 20:00:00,408.767539


# Model Definition


## Rotation And Geometry Constants


In [42]:
rotation = compute_rotation_state(
    cr_days=cr_days,
    phi_step_minutes=phi_step_minutes,
)
cr_time = rotation.cr_time
omega = rotation.omega
phi_step = rotation.phi_step

print("omega * 3600:", omega * 3600)
print("phi_step_used (deg):", phi_step, "| from phi_step_minutes:", phi_step_minutes)
print(
    "time_step_used (h):",
    time_step_hours,
    "| freq:",
    time_freq,
    "| superresolution:",
    superresolution_enabled,
)
print("r0:", r0, "| rSolar(km):", rSolar, "| R_max:", R_max)

omega * 3600: 0.5555555555555556
phi_step_used (deg): 1.1111111111111112 | from phi_step_minutes: 120
time_step_used (h): 1.0 | freq: 60min | superresolution: False
r0: 20 | rSolar(km): 700000.0 | R_max: 215


## Grid Construction (time, phi, R)


In [43]:
grid = build_grid_axes(
    sim_start=sim_start,
    sim_end=sim_end,
    time_freq=time_freq,
    phi_step=phi_step,
    r0=r0,
    r_max=R_max,
    dense_memory_budget_gb=dense_memory_budget_gb,
    memory_guard_enabled=memory_guard_enabled,
)

time_axis = grid.time_axis
phi_axis = grid.phi_axis
r_axis = grid.r_axis
n_cells = grid.n_cells
est_runtime_gb = grid.est_runtime_gb

print("Grid shape (t, phi, R):", (len(time_axis), len(phi_axis), len(r_axis)))
print(
    f"Estimated dense propagation memory: {est_runtime_gb:.2f} GB"
)

Grid shape (t, phi, R): (1640, 324, 196)
Estimated dense propagation memory: 0.67 GB


In [44]:
transport = build_transport_state(
    time_axis=time_axis,
    phi_axis=phi_axis,
    rotation_state=rotation,
    horizon_hours=horizon_hours,
    time_step_hours=time_step_hours,
    field_half_width_h=field_half_width_h,
    r_solar_km=rSolar,
)

t0_ref = transport.t0_ref
phi_delay_h = transport.phi_delay_h
horizon_steps = transport.horizon_steps
h_step_idx = transport.h_step_idx
h_step_hours = transport.h_step_hours
r_kernel_scale = transport.r_kernel_scale
cr_steps = transport.cr_steps
phi_delay_steps = transport.phi_delay_steps
field_half_width_steps = transport.field_half_width_steps

print("Initialized dense propagation grid.")
print("time range:", time_axis.min(), "->", time_axis.max())
print("phi bins:", len(phi_axis), "R bins:", len(r_axis))

valid_seed_mask = (df_v.index >= time_axis.min()) & (df_v.index <= time_axis.max())
df_v_run = df_v.loc[valid_seed_mask].copy()
print("Seeds in simulation range:", len(df_v_run), "of", len(df_v))

Initialized dense propagation grid.
time range: 2016-12-15 16:00:00 -> 2017-02-21 23:00:00
phi bins: 324 R bins: 196
Seeds in simulation range: 1472 of 1472


## Propagation Functions


In [45]:
packet_p, packet_off, packet_alpha = build_packet_geometry(
    phi_delay_steps=phi_delay_steps,
    field_half_width_steps=field_half_width_steps,
)
acc = init_accumulators(
    n_t=len(time_axis),
    n_p=len(phi_axis),
    n_r=len(r_axis),
    use_cr_reset=use_cr_reset,
)

V_accum_max = acc.V_accum_max
V_accum_cr_idx = acc.V_accum_cr_idx
cr_flat = acc.cr_flat
max_flat = acc.max_flat
dims = (len(time_axis), len(phi_axis), len(r_axis))

(
    seed_times,
    seed_vals,
    v_prev,
    v_next,
    seed_t_idx,
    seed_cr_idx_arr,
    seed_r_idx,
) = prepare_seed_inputs(
    df_v_run=df_v_run,
    cr_steps=cr_steps,
    horizon_steps=horizon_steps,
    time_freq=time_freq,
    t0_ref=t0_ref,
    time_step_hours=time_step_hours,
    r_kernel_scale=r_kernel_scale,
    r0=r0,
)

# Experiments


## Bulk Propagation Across Time


In [46]:
stats = run_bulk_propagation(
    seed_vals=seed_vals,
    v_prev=v_prev,
    v_next=v_next,
    seed_t_idx=seed_t_idx,
    seed_cr_idx_arr=seed_cr_idx_arr,
    seed_r_idx=seed_r_idx,
    h_step_idx=h_step_idx,
    packet_off=packet_off,
    packet_p=packet_p,
    packet_alpha=packet_alpha,
    n_t=len(time_axis),
    n_p=len(phi_axis),
    n_r=len(r_axis),
    max_flat=max_flat,
    cr_flat=cr_flat,
    use_swept_cell=use_swept_cell,
    use_cr_reset=use_cr_reset,
    max_seed_batch=max_seed_batch,
)

filled = stats.filled
total = stats.total
prop_seconds = stats.prop_seconds

print("use_swept_cell:", use_swept_cell)
print("use_cr_reset:", use_cr_reset)
print("Transport: packet width (h):", 2.0 * field_half_width_h)
print(
    "time step (h):",
    time_step_hours,
    "| bins per hour:",
    int(round(1.0 / time_step_hours)),
)
print(
    "Accumulated non-empty cells:",
    filled,
    "/",
    total,
    f"({100.0 * filled / total:.2f}%)",
)
print(f"Propagation runtime: {prop_seconds:.2f} s ({prop_seconds / 60.0:.2f} min)")
print(
    f"Seeds processed: {stats.seeds_processed} | seeds/s: {stats.seeds_per_second:.2f}"
)

2D propagate:   0%|          | 0/6 [00:00<?, ?batch/s]

use_swept_cell: True
use_cr_reset: True
Transport: packet width (h): 1.0
time step (h): 1.0 | bins per hour: 1
Accumulated non-empty cells: 75929778 / 104146560 (72.91%)
Propagation runtime: 0.78 s (0.01 min)
Seeds processed: 1472 | seeds/s: 1892.88


## Post-Processing


In [47]:
slow_sw_speed = empirical.slow_sw_speed(time_axis)

post = postprocess_max_field(
    V_accum_max=V_accum_max,
    slow_sw_speed=slow_sw_speed,
    post_chunk_t=post_chunk_t,
)

V_grid = post.V_grid
model_pred_mask = post.max_model_pred_mask
slow_sw_pred_mask = post.max_slow_sw_pred_mask
post_vlims_raw = post.max_vlims_raw
post_fields = V_grid
post_vlims = post_vlims_raw

pred = int(model_pred_mask.sum())
total = int(V_grid.size)
print(
    "Cells with model prediction (max mode):",
    pred,
    "/",
    total,
    f"({100.0 * pred / total:.2f}%)",
)
print("Predicted slow-SW cells (max mode):", int(slow_sw_pred_mask.sum()))
print("Backfillable empty cells (max mode):", int(total - pred))
print("raw v min/max (max mode):", post_vlims_raw)
print(
    "Display v min/max with backfill300 (max mode):",
    (float(np.nanmin(slow_sw_speed)), post_vlims_raw[1]),
)

Post-processing:   0%|          | 0/13 [00:00<?, ?chunk/s]

Cells with model prediction (max mode): 75929778 / 104146560 (72.91%)
Predicted slow-SW cells (max mode): 1374
Backfillable empty cells (max mode): 28216782
raw v min/max (max mode): (302.95794677734375, 737.8361206054688)
Display v min/max with backfill300 (max mode): (286.4670833333335, 737.8361206054688)


# Analysis


## Visualization


## Satellite Data


In [48]:
stereo_parquet_path = Path(
    "/Users/aosh/Developer/Shock-and-Awe/Data/2022-11-24 19-10-00/STEREO.parquet"
)
earth_radii_per_solar_radius = 109.0763707060096

stereo_a_df = pd.read_parquet(
    stereo_parquet_path,
    columns=[
        "N_p",
        "V",
        "T_p",
        "radialDistance",
        "heliographicLatitude",
        "heliographicLongitude",
    ],
).copy()
stereo_a_df.index = pd.to_datetime(stereo_a_df.index, utc=True).tz_convert(None)
stereo_a_df = stereo_a_df.loc[time_axis.min() : time_axis.max()]
stereo_a_df = stereo_a_df.rename(
    columns={
        "N_p": "n_p",
        "V": "v",
        "T_p": "t_p",
        "radialDistance": "r_target",
        "heliographicLatitude": "lat_hge",
        "heliographicLongitude": "phi_target",
    }
)
for column in stereo_a_df.columns:
    stereo_a_df[column] = pd.to_numeric(stereo_a_df[column], errors="coerce")
stereo_a_df["r_target"] = stereo_a_df["r_target"] / earth_radii_per_solar_radius
stereo_a_df = stereo_a_df.resample(time_freq).mean().reindex(time_axis)
stereo_a_df.attrs["sat"] = "stereo_a"
stereo_a_df.attrs["label"] = "STEREO-A"
stereo_a_df.attrs["coord_frame"] = "HGE"
stereo_a_df


,n_p,v,t_p,r_target,lat_hge,phi_target
2016-12-15 16:00:00,3.369000,411.875556,47788.301887,206.434101,5.1,223.300003
2016-12-15 17:00:00,4.562478,419.523333,63571.034483,206.434101,5.1,223.349791
2016-12-15 18:00:00,4.536589,418.822222,46889.122807,206.434101,5.1,223.399994
2016-12-15 19:00:00,4.287133,419.205556,59538.518519,206.434101,5.1,223.399994
2016-12-15 20:00:00,4.433111,415.726667,53616.333333,206.434101,5.1,223.449789
...,...,...,...,...,...,...
2017-02-21 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-02-21 20:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-02-21 21:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-02-21 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN


## Interactive Exploration


In [49]:
plot_sats = [
    {
        "sat": "ace_earth",
        "label": "ACE @ Earth",
        "phi_target": ballistic["earth_phi_target"],
        "r_target": ballistic["earth_r_target"],
    },
    {
        "sat": "stereo_a",
        "label": "STEREO-A",
        "phi_target": 0.0,
        "r_target": ballistic["earth_r_target"],
    },
]

satellite_frames = {
    "ace_earth": load_ace_earth_frame(),
    "stereo_a": stereo_a_df,
}
satellite_swx_frames = {"ace_earth": build_ace_earth_swx_frame(sdo_input_df)}
comparison_frames = {}
for sat_spec in plot_sats:
    sat_name = sat_spec["sat"]
    df_sat = satellite_frames[sat_name].copy()
    df_sat.attrs["label"] = sat_spec["label"]
    comparison_frames[sat_name] = build_satellite_comparison_frame(
        time_axis=time_axis,
        phi_axis=phi_axis,
        r_axis=r_axis,
        grid_raw=V_grid,
        slow_sw_pred_mask=slow_sw_pred_mask,
        slow_sw_speed=slow_sw_speed,
        df_sat=df_sat,
        df_swx=satellite_swx_frames.get(sat_name),
        phi_target=sat_spec["phi_target"],
        r_target=sat_spec["r_target"],
        draw_slow_sw=True,
        backfill_empty_with_300=False,
    )

for sat_spec in plot_sats:
    sat_name = sat_spec["sat"]
    df_sat = satellite_frames[sat_name]
    print(
        sat_spec["label"],
        "rows:",
        len(df_sat),
        "range:",
        df_sat.index.min(),
        "->",
        df_sat.index.max(),
    )


ACE @ Earth rows: 106753 range: 2010-08-01 00:00:00 -> 2024-01-01 00:00:00
STEREO-A rows: 1640 range: 2016-12-15 16:00:00 -> 2017-02-21 23:00:00


In [ ]:
date_strs = [d.strftime("%Y-%m-%d %H:%M:%S") for d in time_axis]
print(
    f"Earth mapping check: R=215 -> r_idx={np.argmin(np.abs(r_axis - ballistic['earth_r_target']))}"
)


def plot_polar(date_str):
    plot_polar_snapshot(
        date_str=date_str,
        time_axis=time_axis,
        phi_axis=phi_axis,
        r_axis=r_axis,
        grid_raw=V_grid,
        post_vlims_raw=post_vlims_raw,
        slow_sw_pred_mask=slow_sw_pred_mask,
        time_step_hours=time_step_hours,
        slow_sw_speed=slow_sw_speed,
        r0=r0,
        comparison_frames=comparison_frames,
        draw_slow_sw=True,
        backfill_empty_with_300=False,
    )


interact(
    plot_polar,
    date_str=widgets.SelectionSlider(
        options=date_strs,
        value=date_strs[0],
        description="date",
        continuous_update=False,
        layout=widgets.Layout(width="90%"),
    ),
)


Earth mapping check: R=215 -> r_idx=195


interactive(children=(SelectionSlider(continuous_update=False, description='date', layout=Layout(width='90%'),…

<function __main__.plot_polar(date_str, draw_slow_sw, backfill_empty_with_300)>

### Animation Export


In [ ]:
anim_target_speedup_vs_baseline = 10.0
anim_baseline_fps = 12.0
anim_fps = 30
anim_speedup_multiplier = 2.0
anim_superres_reference_minutes = 10.0
anim_dpi = runtime["animation_dpi"]
anim_outfile = Path("Outputs/SW/SW Polar Animation.mp4")

primary_sat = plot_sats[0]["sat"]
primary_frame_anim = comparison_frames[primary_sat]
animation_stats = export_polar_animation(
    anim_outfile=anim_outfile,
    time_axis=time_axis,
    phi_axis=phi_axis,
    r_axis=r_axis,
    grid_raw=V_grid,
    post_vlims_raw=post_vlims_raw,
    slow_sw_pred_mask=slow_sw_pred_mask,
    comparison_frames=comparison_frames,
    time_step_minutes=time_step_minutes,
    superresolution_enabled=superresolution_enabled,
    slow_sw_speed=slow_sw_speed,
    r0=r0,
    cr_days=cr_days,
    draw_slow_sw=True,
    backfill_empty_with_300=False,
    target_speedup_vs_baseline=anim_target_speedup_vs_baseline,
    baseline_fps=anim_baseline_fps,
    anim_fps=anim_fps,
    speedup_multiplier=anim_speedup_multiplier,
    superres_reference_minutes=anim_superres_reference_minutes,
    anim_dpi=anim_dpi,
    anim_plot_style="mesh",
)
print(animation_stats)


# Outputs


In [ ]:
# out_dir = Path("Outputs/SW")
# out_dir.mkdir(parents=True, exist_ok=True)
# out_parquet = out_dir / "SW Earth Series.parquet"

# comparison_frames[primary_sat].loc[time_axis.min() : time_axis.max()].to_parquet(out_parquet)
# print("Saved", out_parquet)


In [53]:
# earth_phi_target = float(ballistic["earth_phi_target"])
# earth_r_target = float(ballistic["earth_r_target"])
# phi0_idx = int(np.argmin(np.abs(phi_axis - earth_phi_target)))
# r215_idx = int(np.argmin(np.abs(r_axis - earth_r_target)))

# propagated_earth = pd.Series(
#     V_grid[:, phi0_idx, r215_idx],
#     index=time_axis,
#     name=f"v_propagated_r{int(earth_r_target)}_phi{int(earth_phi_target)}",
# )


# def classify_gap_transition(left_speed, right_speed):
#     if not np.isfinite(left_speed) or not np.isfinite(right_speed):
#         return "edge-missing"
#     if right_speed > left_speed:
#         return "lower -> higher"
#     if right_speed < left_speed:
#         return "higher -> lower"
#     return "equal -> equal"


# gap_mask = propagated_earth.isna()
# gap_groups = gap_mask.ne(gap_mask.shift().fillna(False)).cumsum()
# gap_rows = []

# for _, gap in propagated_earth[gap_mask].groupby(gap_groups[gap_mask]):
#     gap_start = gap.index[0]
#     gap_end = gap.index[-1]
#     start_i = propagated_earth.index.get_loc(gap_start)
#     end_i = propagated_earth.index.get_loc(gap_end)

#     left_i = start_i - 1 if start_i > 0 else None
#     right_i = end_i + 1 if end_i + 1 < len(propagated_earth) else None

#     left_time = propagated_earth.index[left_i] if left_i is not None else pd.NaT
#     right_time = propagated_earth.index[right_i] if right_i is not None else pd.NaT
#     left_speed = propagated_earth.iloc[left_i] if left_i is not None else np.nan
#     right_speed = propagated_earth.iloc[right_i] if right_i is not None else np.nan
#     transition = classify_gap_transition(left_speed, right_speed)

#     gap_rows.append(
#         {
#             "gap_start": gap_start,
#             "gap_end": gap_end,
#             "gap_steps": int(len(gap)),
#             "gap_hours": float(len(gap) * time_step_hours),
#             "left_time": left_time,
#             "left_speed": left_speed,
#             "right_time": right_time,
#             "right_speed": right_speed,
#             "speed_delta": right_speed - left_speed,
#             "transition": transition,
#         }
#     )

# gap_details = pd.DataFrame(gap_rows)

# if gap_details.empty:
#     print("No propagated gaps found at (R=215, phi=0).")
# else:
#     gap_summary = (
#         gap_details.groupby("transition", dropna=False)
#         .agg(
#             gap_count=("transition", "size"),
#             total_gap_hours=("gap_hours", "sum"),
#             median_gap_hours=("gap_hours", "median"),
#         )
#         .sort_values(["gap_count", "total_gap_hours"], ascending=False)
#     )

#     print(
#         f"Propagated gaps at (R={int(earth_r_target)}, phi={int(earth_phi_target)}):",
#         len(gap_details),
#     )
#     print(gap_summary)
#     gap_details = gap_details.sort_values("gap_start").reset_index(drop=True)
#     gap_details